Read data from the books.csv datafile

In [1]:
import pandas as pd

df = pd.read_csv('data\\books.csv',on_bad_lines='skip')

In [2]:
df

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic
...,...,...,...,...,...,...,...,...,...,...,...,...
11118,45631,Expelled from Eden: A William T. Vollmann Reader,William T. Vollmann/Larry McCaffery/Michael He...,4.06,1560254416,9781560254416,eng,512,156,20,12/21/2004,Da Capo Press
11119,45633,You Bright and Risen Angels,William T. Vollmann,4.08,0140110879,9780140110876,eng,635,783,56,12/1/1988,Penguin Books
11120,45634,The Ice-Shirt (Seven Dreams #1),William T. Vollmann,3.96,0140131965,9780140131963,eng,415,820,95,8/1/1993,Penguin Books
11121,45639,Poor People,William T. Vollmann,3.72,0060878827,9780060878825,eng,434,769,139,2/27/2007,Ecco


In [3]:
publisher_lst = df.publisher.unique()

Top five books from a given publisher

In [4]:
topfive = pd.DataFrame()
topfive.columns = []
def top_five(df, publisher_name):
    df_subset = df[df.publisher==publisher_name].sort_values('average_rating', ascending=False)
    df_subset = df_subset[['title', 'average_rating']]
    return df_subset.head(5)

top_five(df, 'Wiley') #Example usage witrh publisher Wiley

,title,average_rating
82,eBay: Top 100 Simplified Tips & Tricks,4.27
4607,Color Drawing: Design Drawing Skills and Techn...,4.26
9767,Grand Finales: A Modernist View of Plated Dess...,4.24
1058,The Science of Sherlock Holmes: From Baskervil...,4.22
8249,Silent Running,4.17


Popularity-based Recommender

Weighted ranking similar to the IMDB rating system

In [5]:
#Define the minimum vote count
minimum_vote_count = 0.75* df['ratings_count'].max()
#Define C – the mean rating
mean_rating = df['average_rating'].mean()

df['weighted_rating'] = (((df['ratings_count']/(df['ratings_count']+minimum_vote_count)) * df['average_rating']) +                   ((minimum_vote_count/(df['ratings_count']+minimum_vote_count))*mean_rating))

recommendations = df.sort_values(by = 'weighted_rating',ascending = False).head(5) # Example usage

Content-based recommender

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


tfidf = TfidfVectorizer()
df.authors = df.authors.fillna('')
tfidf_mat = tfidf.fit_transform(df.authors)
distance_mat = cosine_similarity(tfidf_mat)

In [7]:
indices = pd.Series(df.index, index=df.title).drop_duplicates()
# print(indices)

def ContentBasedRecommender(title):
    id_ = indices[title]
    distances = list(enumerate(distance_mat[id_]))
    distances = sorted(distances,key=lambda x: x[1], reverse=True)
    distances = distances[1:10] #ten most similar books
    recommendations = [distance[0] for distance in distances]
    return df.title.iloc[recommendations]


print(ContentBasedRecommender("In a Sunburned Country")) # Example usage with book title "In a Sunburned Country"


13                            Bill Bryson's African Diary
14      Bryson's Dictionary of Troublesome Words: A Wr...
15                                 In a Sunburned Country
16      I'm a Stranger Here Myself: Notes on Returning...
17      The Lost Continent: Travels in Small Town America
18              Neither Here nor There: Travels in Europe
19                              Notes from a Small Island
20      The Mother Tongue: English and How It Got That...
1161                 A Short History of Nearly Everything
Name: title, dtype: object
